# Binary Sentiment Classification with BERT

Fine-tuning a pre-trained BERT model for binary sentiment classification on movie reviews.

BERT (Bidirectional Encoder Representations from Transformers) is an encoder-only model, which makes it well-suited for classification tasks. Unlike decoder models (like GPT), BERT processes the full input bidirectionally, building a rich contextual representation of each token. For classification, we take the representation of the `[CLS]` token and pass it through a linear layer to produce class logits.

The approach:
1. Tokenize the raw text using BERT's WordPiece tokenizer
2. Wrap the data in a PyTorch `Dataset` / `DataLoader`
3. Load `BertForSequenceClassification` (BERT + a classification head)
4. Fine-tune on our labelled data

In [2]:
dataset = [
    # Positive (25)
    ("This movie is a masterpiece!", "Positive"),
    ("Terrific!", "Positive"),
    ("Absolutely loved every minute of it.", "Positive"),
    ("A stunning achievement in modern cinema.", "Positive"),
    ("The performances were heartfelt and unforgettable.", "Positive"),
    ("Brilliant writing and direction from start to finish.", "Positive"),
    ("I laughed, I cried, I want to see it again.", "Positive"),
    ("Easily the best film I've seen this year.", "Positive"),
    ("Visually breathtaking and emotionally rich.", "Positive"),
    ("A rare gem that lingers long after the credits roll.", "Positive"),
    ("Sharp, funny, and surprisingly moving.", "Positive"),
    ("The lead actor absolutely carries this film.", "Positive"),
    ("Tight pacing and a satisfying ending.", "Positive"),
    ("Genuinely one of the most original stories I've seen in years.", "Positive"),
    ("A triumph on every level.", "Positive"),
    ("Charming, clever, and full of heart.", "Positive"),
    ("The cinematography alone is worth the ticket.", "Positive"),
    ("It's flawed in places, but its ambition more than makes up for it.", "Positive"),
    ("I went in skeptical and walked out a fan.", "Positive"),
    ("A masterclass in tension and atmosphere.", "Positive"),
    ("Funny without trying too hard, and surprisingly smart.", "Positive"),
    ("The chemistry between the leads is electric.", "Positive"),
    ("Beautifully shot and impeccably acted.", "Positive"),
    ("Hands down a must-watch.", "Positive"),
    ("It nails the landing in a way most films don't even attempt.", "Positive"),

    # Negative (25)
    ("Not worth watching!", "Negative"),
    ("A complete waste of two hours.", "Negative"),
    ("Painfully slow and badly written.", "Negative"),
    ("I wanted my money back halfway through.", "Negative"),
    ("The plot makes absolutely no sense.", "Negative"),
    ("Wooden acting and lifeless dialogue.", "Negative"),
    ("One of the worst films I've ever sat through.", "Negative"),
    ("Boring, predictable, and way too long.", "Negative"),
    ("The trailer was better than the entire movie.", "Negative"),
    ("Generic, soulless, and forgettable.", "Negative"),
    ("Bad pacing ruins what could have been a decent idea.", "Negative"),
    ("Skip it. Seriously.", "Negative"),
    ("The script feels like a first draft nobody bothered to revise.", "Negative"),
    ("Visually flat and emotionally empty.", "Negative"),
    ("I checked my phone more than I watched the screen.", "Negative"),
    ("A confused mess of half-baked ideas.", "Negative"),
    ("Even the soundtrack couldn't save this one.", "Negative"),
    ("The ending is so bad it retroactively ruins the rest.", "Negative"),
    ("Oh sure, another two hours I'll never get back.", "Negative"),
    ("Charmless, joyless, and shockingly dull.", "Negative"),
    ("I have never been so bored in a theater.", "Negative"),
    ("Cliché after cliché, with nothing new to offer.", "Negative"),
    ("It tries so hard to be clever and fails every single time.", "Negative"),
    ("Don't believe the hype - this one's a dud.", "Negative"),
    ("Forgettable the moment the credits roll.", "Negative"),
]

## Dataset

A small hand-crafted dataset of 50 movie reviews (25 positive, 25 negative). With only 50 examples we're relying heavily on the pre-trained knowledge already in BERT - fine-tuning just adapts the classification head and nudges the representations toward our task.

In [4]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

/Users/joewilkinson/Projects/whyyy/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Tokenization

BERT uses a WordPiece tokenizer that splits text into subword tokens from a fixed vocabulary. Each input is wrapped with special tokens: `[CLS]` at the start (whose final hidden state is used for classification) and `[SEP]` to mark the end of a segment. We pad all sequences to a uniform length so they can be batched together as tensors.

In [5]:
max_length = 128
formatted_data = [(f"[CLS] {text} [SEP]", label) for text, label in dataset]
tokenized_data = tokenizer(formatted_data, padding=True, truncation=True, max_length=max_length, return_tensors='pt')

In [8]:
import torch
from sklearn.preprocessing import LabelEncoder

input_ids = tokenized_data['input_ids']
attention_mask = tokenized_data['attention_mask']
labels = torch.tensor(LabelEncoder().fit_transform([label for _, label in dataset]))

## Labels and Train/Val Split

We encode the string labels ("Positive"/"Negative") as integers and split 90/10 for training and validation. The attention mask tells BERT which tokens are real content vs. padding so the self-attention layers don't attend to pad tokens.

In [9]:
from sklearn.model_selection import train_test_split

train_inputs, val_inputs, train_labels, val_labels, train_mask, val_mask = train_test_split(
    input_ids, labels, attention_mask, random_state=42, test_size=0.1
)

## Dataset and DataLoader

PyTorch's `Dataset` and `DataLoader` abstractions handle batching and shuffling. The `Dataset` maps an index to a single example (input IDs, attention mask, label), and the `DataLoader` groups them into batches for efficient training.

In [10]:
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {'input_ids': self.input_ids[idx],
                'attention_mask':
                 self.attention_mask[idx],
                'labels': self.labels[idx]}

In [12]:
from torch.utils.data import DataLoader

batch_size = 4
train_dataset = CustomDataset(train_inputs, train_mask, train_labels)
train_dataloader = DataLoader(train_dataset,batch_size=batch_size, shuffle=True)

val_dataset = CustomDataset(val_inputs, val_mask, val_labels)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(set(labels)))

## Model

`BertForSequenceClassification` is BERT with a linear classification head on top. Loading from `bert-base-uncased` gives us the pre-trained encoder weights, but the classification head (`classifier.weight`, `classifier.bias`) is randomly initialized - those are the "MISSING" keys in the load report. The "UNEXPECTED" keys are from BERT's masked language modelling head, which we don't need for classification and can be safely ignored.